In [ ]:
# !pip install pymongo

from pymongo import MongoClient
import pandas as pd
from configparser import ConfigParser
from urllib.parse import quote_plus

In [ ]:
config = ConfigParser()
config.read('config.ini')

In [ ]:
config_env = 'dev'

In [ ]:
host = config.get(config_env, 'host')
port = config.get(config_env, 'port')
user = config.get(config_env, 'user')
password = quote_plus(config.get(config_env, 'password'))
database = config.get(config_env, 'database')
collection_name = config.get(config_env, 'collection_name')

In [ ]:
print(f"mongodb://{user}:{password}@{host}:{port}/")

In [ ]:
mongo_url = f"mongodb://{user}:{password}@{host}:{port}/"
client = MongoClient(mongo_url)
db = client[database]


In [ ]:
collection = db[collection_name]
data = list(collection.find({}))
mongo_data = pd.DataFrame(data)
df = pd.DataFrame(data)

In [ ]:
""" Avaibility Data Cleanup """
availability_df = pd.json_normalize(df['availability'])
availability_df['airbnb_id'] = df['_id']

In [ ]:
""" Images Data Cleanup """
images_df = pd.json_normalize(df['images'])
images_df['airbnb_id'] = df['_id']

In [ ]:
""" Address Data Cleanup """
address_df = pd.json_normalize(df['address'])
address_df['airbnb_id'] = df['_id']

In [ ]:
""" Host Data Cleanup """
host_df = pd.json_normalize(df['host'])
df['host_id'] = pd.json_normalize(df['host'])['host_id']

In [ ]:
""" Review Data Cleanup """
exploed_df = df['reviews'].explode('reviews')
data_series = exploed_df.dropna()
reviews_df = pd.DataFrame(data_series.tolist()) 

In [ ]:
""" Review Score Data Cleanup """
review_scores_df = pd.json_normalize(df['review_scores'])
review_scores_df['airbnb_id'] = df['_id']

In [ ]:
df.drop(columns=['host', 'reviews', 'address', 'images', 'availability'], inplace=True)

In [ ]:
reviews_df.to_csv(f'data/reviews_df.csv', index=False)
availability_df.to_csv(f'data/availability_df.csv', index=False)
address_df.to_csv(f'data/address_df.csv', index=False)
host_df.to_csv(f'data/host_df.csv', index=False)
review_scores_df.to_csv(f'data/review_scores_df.csv', index=False)
df.to_csv(f'data/main_df.csv', index=False)